# London Fire Brigade Analysis

This notebook analyzes the London fire brigade event timestamps with the same Poisson-compatible window-selection workflow used for earlier datasets.

We parse the timestamps, aggregate them into hourly counts, and check whether the count stream is suitable for the non-hurdle Poisson-style window selector.


In [2]:
from collections import defaultdict
from datetime import datetime
from pathlib import Path
import csv
import sys

ROOT = Path.cwd().resolve()
candidate_roots = [ROOT, *ROOT.parents]
for candidate in candidate_roots:
    if (candidate / "src" / "count_ts_window_selector").exists():
        ROOT = candidate
        break

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

DATA_FILE = ROOT / "data" / "london-fire-brigade_timestamp_events.csv"
print(f"Reading {DATA_FILE}")

timestamps = []
with DATA_FILE.open() as f:
    reader = csv.DictReader(f)
    for row in reader:
        raw = row["timestamp"]
        if "." in raw:
            base, frac = raw.split(".", 1)
            raw = base + "." + frac[:6]
        timestamps.append(datetime.strptime(raw, "%Y-%m-%d %H:%M:%S.%f"))

print(f"Rows: {len(timestamps)}")
print(f"Minimum timestamp: {min(timestamps)}")
print(f"Maximum timestamp: {max(timestamps)}")
print(f"Time span (hours): {(max(timestamps) - min(timestamps)).total_seconds() / 3600:.2f}")


Reading /home/rajiv/programming/count_ts_window_selector/data/london-fire-brigade_timestamp_events.csv
Rows: 6484
Minimum timestamp: 2024-01-01 08:00:18.955096
Maximum timestamp: 2024-01-30 19:59:06.648951
Time span (hours): 707.98


In [3]:
hourly_counts = defaultdict(int)
for dt in timestamps:
    key = dt.replace(minute=0, second=0, microsecond=0)
    hourly_counts[key] += 1

series = [hourly_counts[k] for k in sorted(hourly_counts)]
print(f"Hourly bins: {len(series)}")
print(f"Mean hourly count: {sum(series) / len(series):.2f}")
print(f"Min hourly count: {min(series)}")
print(f"Max hourly count: {max(series)}")
print(f"Zero-hour proportion: {sum(v == 0 for v in series) / len(series):.3f}")
print("First 20 hourly counts:", series[:20])


Hourly bins: 360
Mean hourly count: 18.01
Min hourly count: 9
Max hourly count: 31
Zero-hour proportion: 0.000
First 20 hourly counts: [22, 18, 11, 19, 14, 17, 20, 24, 29, 22, 19, 16, 12, 21, 20, 25, 12, 18, 17, 19]


In [4]:
from count_ts_window_selector import (
    WindowSelectionConfig,
    check_data_suitability,
    select_time_window,
)

config = WindowSelectionConfig(
    min_event_rate=5.0,
    max_zero_window_proportion=0.1,
    min_windows=30,
    confidence_level=0.90,
)

print("Suitability check:", check_data_suitability(series, min_event_rate=config.min_event_rate, max_zero_window_proportion=config.max_zero_window_proportion))

candidate_windows = [1, 2, 3, 4, 6, 8, 12, 24]
results = select_time_window(series, candidate_windows, config=config)
print("Selected Poisson-compatible windows:")
for result in results:
    print(result)


Suitability check: {'observed_event_rate': 18.011111111111113, 'zero_window_proportion': 0.0}
Selected Poisson-compatible windows:
{'window_size': 1, 'fano_factor': 0.9848386171059165, 'fano_lower': 0.8804770289246303, 'fano_upper': 1.1258554818849202}
{'window_size': 2, 'fano_factor': 1.0344190598947471, 'fano_lower': 0.8327133864631102, 'fano_upper': 1.179983137687625}
{'window_size': 3, 'fano_factor': 0.9517726893348333, 'fano_lower': 0.7967330835235972, 'fano_upper': 1.2223591615526457}
{'window_size': 4, 'fano_factor': 1.0757751145429717, 'fano_lower': 0.7668458825904587, 'fano_upper': 1.2586739971888523}
{'window_size': 6, 'fano_factor': 1.1285772540490804, 'fano_lower': 0.7176153852561603, 'fano_upper': 1.3208563356818717}
{'window_size': 8, 'fano_factor': 1.1734619482922997, 'fano_lower': 0.6769881154741354, 'fano_upper': 1.3745656041440102}
{'window_size': 12, 'fano_factor': 0.9715373651853901, 'fano_lower': 0.6106333166491235, 'fano_upper': 1.467481648423886}


## Interpretation

The London fire brigade event stream is dense enough for the non-hurdle workflow, with a mean hourly rate of about 18.0 events/hour and no zero-hour windows. The Fano factors remain near 1 across several candidate windows, which indicates a relatively stable near-Poisson arrival process over short to moderate aggregation horizons.

The 1-hour window is the closest match to the Poisson reference of 1, making it the best default observation window for this dataset.


## Visual diagnostics

The next cells plot the hourly event counts and the Fano factor as a function of aggregation window.


In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

hour_labels = sorted(hourly_counts)
hour_values = [hourly_counts[k] for k in hour_labels]
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(hour_labels, hour_values, color="#2f70af", linewidth=1.5)
ax.set_title("Hourly London fire brigade event counts")
ax.set_xlabel("Hour bucket")
ax.set_ylabel("Events")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [8]:
import numpy as np

config = WindowSelectionConfig(
    min_event_rate=5.0,
    max_zero_window_proportion=0.1,
    min_windows=30,
    confidence_level=0.90,
)
results = select_time_window(series, [1, 2, 3, 4, 6, 8, 12, 24], config=config)
windows = [r["window_size"] for r in results]
fanos = [r["fano_factor"] for r in results]
lower = [r["fano_lower"] for r in results]
upper = [r["fano_upper"] for r in results]

fig, ax = plt.subplots(figsize=(8, 5))
yerr = [np.array(fanos) - np.array(lower), np.array(upper) - np.array(fanos)]
ax.errorbar(windows, fanos, yerr=yerr, fmt="o-", color="#2f70af", capsize=4)
ax.axhline(1.0, linestyle="--", color="#d65f5f", linewidth=1.2, label="Poisson reference (Fano = 1)")
ax.set_title("Fano factor vs. aggregation window")
ax.set_xlabel("Window size (hours)")
ax.set_ylabel("Fano factor")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()
